In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, sum, avg, count, broadcast, row_number, rank, dense_rank, lag, lead, coalesce, lit, expr, asc, desc
from pyspark.sql.window import Window
from pyspark.core.context import SparkContext

spark = SparkSession.builder \
    .appName("ExamenPySpark") \
    .master("local[*]") \
    .getOrCreate()

sc : SparkContext = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 09:03:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.parquet("data/features/training_dataset")

In [4]:
df = (
    df
    .filter(col("genres").isNotNull() & expr("size(genres) > 0"))
    .withColumn("genre", col("genres")[0])
    .drop("genres")
)

df.show()

26/03/24 09:03:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+--------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+-------------------+----------

In [5]:
genre2id = (
    df
    .select("genre")
    .distinct()
    .withColumn("genre_id", row_number().over(Window.orderBy("genre")) - 1)
).collect()
genre2id_dict = {row['genre']: row['genre_id'] for row in genre2id}
genre2id_broadcast = sc.broadcast(genre2id_dict)
genre2id_dict

26/03/24 09:03:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 09:03:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 09:03:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 09:03:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 09:03:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 09:03:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 0

{'60s': 0,
 '70s': 1,
 '80s': 2,
 '8bit': 3,
 '90s': 4,
 'acid': 5,
 'acidhouse': 6,
 'acidjazz': 7,
 'acousticrock': 8,
 'african': 9,
 'alternative': 10,
 'alternativerock': 11,
 'ambiant': 12,
 'ambient': 13,
 'ambiente': 14,
 'americana': 15,
 'artrock': 16,
 'asian': 17,
 'atmospheric': 18,
 'avantgarde': 19,
 'baroque': 20,
 'bebop': 21,
 'blackmetal': 22,
 'blues': 23,
 'bluesrock': 24,
 'bossanova': 25,
 'breakbeat': 26,
 'britpop': 27,
 'cabaret': 28,
 'celtic': 29,
 'chanson': 30,
 'chansonfrancaise': 31,
 'chillout': 32,
 'chilloutlounge': 33,
 'choir': 34,
 'choral': 35,
 'christian': 36,
 'classical': 37,
 'classicalmusic': 38,
 'classicrock': 39,
 'club': 40,
 'contemporary': 41,
 'country': 42,
 'dance': 43,
 'dancehall': 44,
 'dancepop': 45,
 'darkambient': 46,
 'darkwave': 47,
 'death': 48,
 'deathmetal': 49,
 'deephouse': 50,
 'disco': 51,
 'doom': 52,
 'downbeat': 53,
 'downtempo': 54,
 'dreampop': 55,
 'drumnbass': 56,
 'dub': 57,
 'dubelectro': 58,
 'dubstep': 59,


In [6]:
# add genre_id column to df
genre2id_df = spark.createDataFrame(genre2id)  # columns: genre, genre_id

df = df.join(
    broadcast(genre2id_df),
    on="genre",
    how="left"
)

df.show()

+-------------+---------+--------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+----------------

In [7]:

X_cols = df.columns
# remove track_num; path; meta_path; btach_id; track_id; genres
X_cols = [col for col in X_cols if col not in ['track_num', 'path', 'meta_path', 'batch_id', 'track_id', 'TRACK_ID','genres','genre']]

X_df = df.select(X_cols)
X_df.show()

+-------------------+------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+--

In [27]:
# keep only top 10 genres
top_genres = (
    df
    .groupBy("genre")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
    .limit(10)
    .select("genre")
    .rdd.flatMap(lambda x: x)
    .collect()
)
df = df.filter(col("genre").isin(top_genres))
df.show()

+-------------+---------+--------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+--------------------+------------------+-------------------+------------------+--------------

In [8]:
import lightgbm as lgb


In [28]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight


# Avoid label leakage: remove genre_id from features
feature_cols = [c for c in X_cols if c != "genre_id"]

# Prepare training data
train_pdf = (
    df.select(*feature_cols, col("genre_id").cast("int").alias("label"))
      .dropna()
      .toPandas()
)

X = train_pdf[feature_cols]
y = train_pdf["label"]

# Split strategy:
# - classes with >=2 samples: stratified split
# - classes with 1 sample: keep in train only
class_counts = y.value_counts()
eligible_idx = y[y.map(class_counts) >= 2].index
rare_idx = y[y.map(class_counts) < 2].index

train_idx, valid_idx = train_test_split(
    eligible_idx,
    test_size=0.2,
    random_state=42,
    stratify=y.loc[eligible_idx]
)

train_idx = list(train_idx) + list(rare_idx)

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_valid, y_valid = X.loc[valid_idx], y.loc[valid_idx]

# Convert to numpy to avoid sklearn feature_names_in_ setter issue
X_train_np = X_train.to_numpy(dtype="float32")
X_valid_np = X_valid.to_numpy(dtype="float32")
y_train_np = y_train.to_numpy()
y_valid_np = y_valid.to_numpy()

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train_np)


# Train LightGBM multiclass model
lgbm_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(genre2id_dict),
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=len(genre2id_dict) * 4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm_model.fit(
    X_train_np,
    y_train_np,
    eval_set=[(X_valid_np, y_valid_np)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
    sample_weight=sample_weights,
)

# Evaluate
y_pred = lgbm_model.predict(X_valid_np)
print("Validation accuracy:", accuracy_score(y_valid_np, y_pred))
print("Validation macro-F1:", f1_score(y_valid_np, y_pred, average="macro"))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001878 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 49597
[LightGBM] [Info] Number of data points in the train set: 1462, number of used features: 207
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning]

In [17]:
from utils.features import pipeline_features

In [30]:
L = pipeline_features("Rape Me.mp3")
L

[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?


{'mfcc_1_mean': -165.2390626116592,
 'mfcc_1_std': 113.56280420495354,
 'mfcc_1_min': -529.8038330078125,
 'mfcc_1_max': -15.69790267944336,
 'mfcc_2_mean': 153.4314155825089,
 'mfcc_2_std': 31.1978927468362,
 'mfcc_2_min': 0.0,
 'mfcc_2_max': 224.84884643554688,
 'mfcc_3_mean': -30.185469046570326,
 'mfcc_3_std': 27.97291147326768,
 'mfcc_3_min': -80.7137451171875,
 'mfcc_3_max': 89.59393310546875,
 'mfcc_4_mean': 33.36083768844039,
 'mfcc_4_std': 27.3646026942444,
 'mfcc_4_min': -47.38890838623047,
 'mfcc_4_max': 84.46505737304688,
 'mfcc_5_mean': 3.1470081155592426,
 'mfcc_5_std': 21.504858024169177,
 'mfcc_5_min': -46.2464599609375,
 'mfcc_5_max': 75.81761932373047,
 'mfcc_6_mean': 31.267316764647404,
 'mfcc_6_std': 12.269265235280576,
 'mfcc_6_min': -5.585150718688965,
 'mfcc_6_max': 67.94527435302734,
 'mfcc_7_mean': -7.714322215178886,
 'mfcc_7_std': 9.613625409323838,
 'mfcc_7_min': -43.55061340332031,
 'mfcc_7_max': 34.322593688964844,
 'mfcc_8_mean': 6.879896010582264,
 'mfcc

In [33]:
# predict genre_id for Lumière.mp3
L_df = spark.createDataFrame([L])
L_df.show()
L_np = L_df.toPandas().to_numpy(dtype="float32")
pred = lgbm_model.predict(L_np)
print(pred)

+-----------------+-----------------+----------------+-----------------+------------------+------------------+------------------+-----------------+-------------+------------------+--------------------+-------------------+-------------+------------------+--------------------+-----------------+-------------+------------------+--------------------+------------------+------------+------------------+------------------+------------------+------------+------------------+--------------------+-------------------+------------+------------------+--------------------+------------------+------------+-------------------+--------------------+------------------+------------+------------------+--------------------+------------------+------------+------------------+--------------------+------------------+------------+------------------+--------------------+------------------+------------+-------------------+--------------------+-------------------+------------+-------------------+--------------------+--